# DIESEL — Notebook 03: Error Analysis (Novel Contribution)
Evaluates Round 1 model, classifies errors using the 6-category taxonomy,
and generates the Difficulty × Error-Type matrix.

**Runtime:** Select the **L4 GPU** kernel.

**Prerequisites:** Round 1 training completed (02_train_round1.ipynb)

In [ ]:
!pip install -q torch transformers peft trl bitsandbytes
!pip install -q accelerate datasets sqlparse sqlglot
!pip install -q wandb matplotlib seaborn scipy scikit-learn python-dotenv tqdm

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# === Colab Setup: Clone project & set path ===
import os, sys

if not os.path.exists('/content/Text-to-SQL-LLM/src'):
    !git clone https://github.com/Deepak-Lingala/Text-to-SQL-LLM.git /content/Text-to-SQL-LLM

os.chdir('/content/Text-to-SQL-LLM')
sys.path.insert(0, '/content/Text-to-SQL-LLM')
print('Project root:', os.getcwd())

import torch, gc, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CONFIGURATION — adjust these paths if needed
# ============================================================
ROUND1_ADAPTER = 'outputs/round1/final_adapter'  # from notebook 02
SPIDER_DB_DIR = 'spider_databases'  # path to Spider SQLite DBs
EVAL_BASE_MODEL = True  # also evaluate base model for comparison
MAX_SAMPLES = None  # set to e.g. 50 for quick test, None for full eval

In [ ]:
from src.config import get_config
from src.data_loader import load_spider_dataset
from src.model_loader import load_finetuned_model, load_base_model, load_tokenizer
from src.evaluate import evaluate_model, compare_models
from src.error_analyzer import analyze_errors, compare_error_distributions, ErrorCategory

config = get_config()
sns.set_theme(style='whitegrid', font_scale=1.1)
FIGURES_DIR = config.paths.figures_dir
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load data
dataset, schema_manager = load_spider_dataset(config)
eval_data = dataset['validation']
if MAX_SAMPLES:
    eval_data = eval_data.select(range(min(MAX_SAMPLES, len(eval_data))))
print(f'Evaluation examples: {len(eval_data)}')

## Step 1: Evaluate Base Model (Zero-Shot)

In [ ]:
all_results = []

if EVAL_BASE_MODEL:
    print('Evaluating BASE model (zero-shot)...')
    base_model = load_base_model(config)
    base_tokenizer = load_tokenizer(config)
    base_model.eval()
    
    base_results = evaluate_model(
        model=base_model, tokenizer=base_tokenizer,
        eval_data=eval_data, schema_manager=schema_manager,
        spider_db_dir=SPIDER_DB_DIR, config=config,
        model_name='base_zero_shot', save_dir=config.paths.eval_dir,
    )
    all_results.append(base_results)
    del base_model; gc.collect(); torch.cuda.empty_cache()

## Step 2: Evaluate Fine-Tuned Model (Round 1)

In [ ]:
print('Evaluating ROUND 1 fine-tuned model...')
ft_model, ft_tokenizer = load_finetuned_model(ROUND1_ADAPTER, config)

ft_results = evaluate_model(
    model=ft_model, tokenizer=ft_tokenizer,
    eval_data=eval_data, schema_manager=schema_manager,
    spider_db_dir=SPIDER_DB_DIR, config=config,
    model_name='round1_finetuned', save_dir=config.paths.eval_dir,
)
all_results.append(ft_results)
del ft_model; gc.collect(); torch.cuda.empty_cache()

## Step 3: Statistical Comparison

In [ ]:
if len(all_results) > 1:
    comparison = compare_models(all_results, save_dir=config.paths.eval_dir)

## Step 4: Error Taxonomy Analysis (NOVEL)

In [ ]:
# Analyze Round 1 errors
ft_error_analysis = analyze_errors(
    eval_results=ft_results,
    schema_map=schema_manager.schema_map,
    config=config,
)

if EVAL_BASE_MODEL:
    base_error_analysis = analyze_errors(
        eval_results=base_results,
        schema_map=schema_manager.schema_map,
        config=config,
    )
    shift = compare_error_distributions(
        base_error_analysis, ft_error_analysis,
        save_dir=config.paths.error_analysis_dir,
    )

## Step 5: Publication Figures

In [ ]:
# Figure 4: Error Category Distribution
cats = ErrorCategory.ALL
labels = [ErrorCategory.LABELS[c] for c in cats]
pcts = [ft_error_analysis['category_distribution'][c]['percentage_of_errors'] for c in cats]

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Set2', len(cats))
bars = ax.bar(labels, pcts, color=colors, edgecolor='gray', linewidth=0.5)
ax.set_ylabel('Percentage of Errors (%)')
ax.set_title('Error Category Distribution — Round 1 Fine-Tuned Model')
ax.set_xticklabels(labels, rotation=30, ha='right')
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_error_distribution.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Difficulty x Error-Type Heatmap (KEY FIGURE)
matrix_data = ft_error_analysis.get('difficulty_error_matrix', {})

if matrix_data:
    difficulties = sorted(matrix_data.keys())
    matrix = np.zeros((len(difficulties), len(cats)))
    for i, diff in enumerate(difficulties):
        for j, cat in enumerate(cats):
            matrix[i, j] = matrix_data[diff].get(cat, 0)
    
    row_sums = matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    matrix_pct = matrix / row_sums * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    cat_labels = [ErrorCategory.LABELS[c] for c in cats]
    
    sns.heatmap(matrix, annot=True, fmt='.0f', cmap='YlOrRd',
                xticklabels=cat_labels, yticklabels=difficulties, ax=axes[0])
    axes[0].set_title('Error Counts: Difficulty × Category')
    
    sns.heatmap(matrix_pct, annot=True, fmt='.1f', cmap='YlOrRd',
                xticklabels=cat_labels, yticklabels=difficulties, ax=axes[1])
    axes[1].set_title('Error Distribution (%): Difficulty × Category')
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'fig5_difficulty_error_heatmap.png'), bbox_inches='tight')
    plt.show()

In [ ]:
# Figure 6: Base vs Fine-Tuned Error Comparison
if EVAL_BASE_MODEL:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(cats))
    width = 0.35
    base_pcts = [base_error_analysis['category_distribution'][c]['percentage_of_errors'] for c in cats]
    ft_pcts = [ft_error_analysis['category_distribution'][c]['percentage_of_errors'] for c in cats]
    
    ax.bar(x - width/2, base_pcts, width, label='Base (Zero-Shot)', color='#FF7043', alpha=0.85)
    ax.bar(x + width/2, ft_pcts, width, label='Round 1 (Fine-Tuned)', color='#42A5F5', alpha=0.85)
    ax.set_ylabel('Percentage of Errors (%)')
    ax.set_title('Error Distribution Shift: Base → Fine-Tuned')
    ax.set_xticks(x)
    ax.set_xticklabels([ErrorCategory.LABELS[c] for c in cats], rotation=30, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'fig6_error_shift.png'), bbox_inches='tight')
    plt.show()

In [ ]:
print('\n' + '=' * 60)
print('Error Analysis Complete!')
print('=' * 60)
print(f'Figures saved to: {FIGURES_DIR}')
print(f'Analysis saved to: {config.paths.error_analysis_dir}')
print(f'\nTop-3 weaknesses for TGDA:')
for i, w in enumerate(ft_error_analysis['ranked_weaknesses'][:3], 1):
    print(f"  {i}. {w['label']}: {w['error_rate']:.1f}% of errors")
print(f'\nNext: Run 04_augment_and_train_round2.ipynb')